# 04. Data Preparation and Combined Features

Flow: load/download data -> clean -> split -> join movie details -> encode the
full prepared DataFrame with `generate_tfidf_folds`.

The shared function now handles every selected feature: TF-IDF for each text
field, genre encoding, numeric imputation/scaling, sparse matrix combination,
and optional training-only oversampling. All fitting happens inside each fold.

## 1. Setup

Start from the repository root or `notebooks/`. The working directory is set
to the repository root because the existing download function uses relative paths.
Restart the kernel after editing imported modules if it still uses older code.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from scipy.sparse import issparse

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start in the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.download_data import download_data
from src.data_cleaning import clean_data
from src.split_data import create_train_test_split, RANDOM_STATE
from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.text_features import generate_tfidf_folds

In [ ]:
group_by = "review_text" 
sample_size = 10000      # Use None for the complete training partition.
remove_spoilers = False
oversample_positive = True
selected_features = INITIAL_FEATURES.copy()

## 2. Download, Clean, and Split

These calls use the existing project methods and skip stages whose files already
exist. Cleaning normalizes movie IDs; it does not remove spoiler cues.

Keep the outer split fixed during modelling. Existing files are not checked for
freshness: if the raw data or split rules change, rerun `clean_data()` and
`create_train_test_split(group_by=group_by)` deliberately before experiments.

In [ ]:
dataset_files = ["IMDB_reviews.json", "IMDB_movie_details.json"]
raw_dir = PROJECT_ROOT / "data/raw"
cleaned_dir = PROJECT_ROOT / "data/cleaned"

if not all((raw_dir / name).exists() for name in dataset_files):
    download_data()
if not all((cleaned_dir / name).exists() for name in dataset_files):
    clean_data()

split_dir = PROJECT_ROOT / "data/splits"
if group_by == "movie_id":
    split_dir = split_dir / "movie_disjoint"
split_files = ["IMDB_reviews_train.json", "IMDB_reviews_test.json"]
if not all((split_dir / name).exists() for name in split_files):
    create_train_test_split(group_by=group_by)

print("Using split:", split_dir)

## 3. Load the Training Split

Use a small training-only sample for a first run, then set `sample_size = None`
for full training-data experiments. The existing `data_loader` functions load
raw data, so here we read the already cleaned metadata and saved training split.
The final test file is not loaded.

In [ ]:
movies = pd.read_json(
    PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json",
    lines=True, convert_dates=False,
)
train_reviews = pd.read_json(
    split_dir / "IMDB_reviews_train.json", lines=True, convert_dates=False,
)

reviews = (
    train_reviews.sample(n=min(sample_size, len(train_reviews)), random_state=RANDOM_STATE)
    if sample_size is not None else train_reviews
).reset_index(drop=True)
print("Development reviews:", len(reviews))

## 4. Prepare One DataFrame

`prepare_features` joins movie details, keeps the review rating, converts duration
to minutes, and computes days since release. Attach the target and grouping ID
so the fold function receives one DataFrame.

Only `selected_features` become model inputs. `is_spoiler` is the target and
`movie_id` is for grouping; neither is encoded. Keep original `review_text` for
grouping even when excluding it from the model.

In [ ]:
model_data = prepare_features(reviews, movies)
model_data["is_spoiler"] = reviews["is_spoiler"].to_numpy()
model_data["movie_id"] = reviews["movie_id"].to_numpy()

assert model_data.index.equals(reviews.index)
print("Prepared data:", model_data.shape)
display(model_data.head())

| Input feature | Processing inside generate_tfidf_folds |
| --- | --- |
| review_text | Independent TF-IDF vocabulary |
| review_summary | Independent TF-IDF vocabulary |
| plot_summary | Independent TF-IDF vocabulary |
| plot_synopsis | Independent TF-IDF vocabulary |
| genre | Binary columns for genres seen in training |
| rating | Training-median imputation and maximum-absolute-value scaling |
| duration | Same numeric processing, in minutes |
| days_since_release | Same numeric processing, in days |

Each field stays a separate feature block. These plot-text features do not yet
measure explicit review-to-plot similarity. Missing text becomes an empty string;
text with no training tokens contributes zero columns. All-missing numeric
training columns are kept and filled with zero.

## 5. Encode the Full DataFrame Within Each Fold

One function now fits all selected encoders on training rows, transforms
validation rows, combines their features, and optionally oversamples complete
training rows. Validation data is never oversampled.

`max_features` caps each text vocabulary separately. Pass `feature_columns`
explicitly; leaving it out retains the older review-text-only behaviour.

In [ ]:
fold_shapes = []
for fold in generate_tfidf_folds(
    model_data,
    feature_columns=selected_features,
    k=5,
    random_state=RANDOM_STATE,
    group_by=group_by,
    remove_spoilers=remove_spoilers,
    oversample_positive=oversample_positive,
    max_features=30000,
):
    X_train, X_val = fold["X_train"], fold["X_val"]
    y_train, y_val = fold["y_train"], fold["y_val"]
    feature_slices = fold["feature_slices"]

    fold_shapes.append({
        "fold": fold["fold"], 
        "train_rows": X_train.shape[0],
        "validation_rows": X_val.shape[0], 
        "features": X_train.shape[1],
        "train_spoiler_rate": y_train.mean(), 
        "validation_spoiler_rate": y_val.mean()
    })

    # Fit and evaluate your model here; record scores before the next fold.

display(pd.DataFrame(fold_shapes))

The summary reports all five folds. After the loop, the matrix variables contain
only the last fold. Fit and evaluate your model inside the loop when adding modelling.

`fold["transformer"]` contains all fitted encoders, including text cleanup.
`fold["feature_slices"]` maps each selected feature to its column range in the
returned matrices. Vocabularies and ranges can differ between folds.
`train_idx` includes duplicates when oversampling, aligned with the training matrix.

## 6. Future Backward Elimination

To exclude `rating`, set
`selected_features = [name for name in INITIAL_FEATURES if name != "rating"]`
and rerun section 5. The shared function then fits only the selected feature
blocks. Keep grouping columns in `model_data`; changing features must not change
fold membership.

To switch ablation on/off, change `remove_spoilers` and rerun section 5. The
function removes cues from review text and review summary without modifying
`model_data`. Compare models using the same folds and sampling settings.

After choosing features and hyperparameters, use `create_feature_transformer`
from `src.text_features` to build fresh encoders on the full prepared training
partition. Fit on training data, transform the prepared test data, and apply
`oversample_positive_indices` only to training rows after encoding.
This notebook does not train models or perform automatic backward elimination.